# DATA304 · 체크포인트로 검색 병목 진단

학습한 ColBERT를 불러와 같은 VAL 쿼리에서 **FAISS 토큰 이웃 → ANN 상품 후보 풀 → MaxSim 순위**를 구합니다. Cross-encoder 체크포인트가 있으면 최종 Top 10까지 평가합니다.

- 학습에 사용한 Python/PyTorch 커널에서 이 파일을 프로젝트 폴더에 두고 실행하세요. 첫 코드 셀에서 `PROJECT_DIR`과 `RUN_DIR`을 맞춥니다.
- **필수**는 `catalog.json`, `train_gold.json`, 그리고 `data304_outputs/checkpoints/colbert_gold.pt`입니다. 기본값은 pseudo 학습을 건너뛴 실행에 맞췄습니다. `cross_gold.pt`와 `split.json`이 있으면 함께 읽습니다. 체크포인트 경로를 직접 지정할 수도 있습니다.
- 같은 체크포인트·카탈로그로 만든 `index_gold`가 있으면 `EXISTING_INDEX_DIR`에 그 경로를 적으면 됩니다. 기본 실행은 체크포인트에서 진단용 인덱스를 한 번 만든 뒤 재사용합니다. GPU에서 처음 빠르게 확인하려면 `VAL_LIMIT = 20`으로 시작하고, 기존 350건 로그와 비교할 때는 `350`을 사용하세요. VAL 쿼리 수를 줄여도 인덱스가 다루는 상품 범위는 전체 카탈로그입니다.
- 모델 추론과 로컬 평가만 필요한 노트북입니다. 토크나이저와 모델 설정은 학습에 사용한 백본의 캐시 또는 `backbone`에 지정한 로컬 경로에서 읽을 수 있습니다.

In [ ]:
# 1. 프로젝트 경로와 진단 범위를 정한다. 기존 학습 폴더의 위치가 다르면 RUN_DIR을 수정한다.
from pathlib import Path

PROJECT_DIR = Path.cwd()
RUN_DIR = PROJECT_DIR / 'data304_outputs'
DATA_DIR = None  # 예: PROJECT_DIR / 'project_release'

CHECKPOINT_STAGE = 'gold'  # pseudo 학습을 완료한 체크포인트를 고르려면 'final'
COLBERT_CHECKPOINT = None  # None이면 RUN_DIR/checkpoints/colbert_gold.pt (또는 colbert_final.pt)
CROSS_CHECKPOINT = None  # None이면 같은 단계의 cross_*.pt가 있을 때 사용
RUN_CROSS = True
EXISTING_INDEX_DIR = None  # 같은 실행의 인덱스를 쓰는 예: RUN_DIR / 'index_gold'
VAL_LIMIT = 350  # 빠른 확인은 20, 전체 VAL은 None
SPLIT_SEED = 42  # split.json이 없을 때 사용한 학습 시드
RUN_ANN_SWEEP = False  # True이면 토큰 이웃 수만 두 배, ANN 풀만 두 배인 경우를 추가 평가
CFG_OVERRIDES = {}  # 예: {'ann_pool': 900, 'candidate_topk': 50}

In [ ]:
# 2. 학습 실행의 설정과 VAL을 복원한다. 패키지가 빠졌으면 학습에 사용한 환경을 선택한다.
# 최초 설치가 필요한 경우 새 셀에서 아래 명령을 실행한다.
# %pip install 'transformers>=4.46,<5' 'faiss-cpu>=1.8' pandas tqdm
import hashlib
import json
import os
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime, timezone
from itertools import islice
from tempfile import TemporaryDirectory

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BFLOAT16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
faiss.omp_set_num_threads(min(8, os.cpu_count() or 4))


def amp_context():
    # GPU가 bfloat16을 지원할 때만 학습 노트북과 같은 혼합 정밀도를 사용한다.
    return torch.autocast(device_type='cuda', dtype=torch.bfloat16) if USE_BFLOAT16 else nullcontext()


def read_json(path):
    # 공식 JSON과 UTF-8 BOM이 붙은 로컬 복사본을 읽는다.
    with Path(path).open(encoding='utf-8-sig') as stream:
        return json.load(stream)


def write_json(path, value):
    # 실행 경로와 요약 지표를 CSV와 나란히 저장한다.
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


RUN_DIR = Path(RUN_DIR).expanduser().resolve()
COLBERT_PATH = Path(COLBERT_CHECKPOINT or RUN_DIR / 'checkpoints' / f'colbert_{CHECKPOINT_STAGE}.pt').expanduser()
CROSS_PATH = Path(CROSS_CHECKPOINT or RUN_DIR / 'checkpoints' / f'cross_{CHECKPOINT_STAGE}.pt').expanduser()
if not COLBERT_PATH.is_file():
    raise FileNotFoundError(f'검색기 체크포인트가 없습니다. 첫 셀의 COLBERT_CHECKPOINT를 지정하세요. 확인한 경로: {COLBERT_PATH}')
if RUN_CROSS and CROSS_CHECKPOINT is not None and not CROSS_PATH.is_file():
    raise FileNotFoundError(f'지정한 CROSS_CHECKPOINT가 없습니다. {CROSS_PATH}')
EVAL_CROSS = RUN_CROSS and CROSS_PATH.is_file()

# 원래 학습 노트북의 기본값 → 저장한 실행 설정 → 이번 진단에서 명시한 설정 순으로 적용한다.
CFG = {'backbone': 'distilbert/distilbert-base-uncased',
 'query_tokens': 72,
 'metadata_tokens': 112,
 'cross_tokens': 448,
 'cross_tokens_per_review': 58,
 'val_fraction': 0.1,
 'ann_nlist': 1024,
 'ann_nprobe': 32,
 'ann_per_token': 48,
 'ann_pool': 450,
 'candidate_topk': 50,
 'cross_infer_batch': 32,
 'index_doc_batch': 96,
 'index_add_docs': 500}
settings_source = '학습 노트북 기본값'
for config_file in (RUN_DIR / 'run_manifest.json', RUN_DIR / 'aspect_vocabulary_and_config.json'):
    if config_file.is_file():
        saved = read_json(config_file).get('config', {})
        if saved:
            CFG.update({key: saved[key] for key in CFG if key in saved})
            settings_source = str(config_file)
            break
unknown_settings = CFG_OVERRIDES.keys() - CFG.keys()
if unknown_settings:
    raise ValueError(f'알 수 없는 CFG_OVERRIDES 키: {sorted(unknown_settings)}')
CFG.update(CFG_OVERRIDES)

# 검증에는 정답이 들어 있는 train_gold.json과 카탈로그만 사용한다.
project = Path(PROJECT_DIR).expanduser().resolve()
release = project / 'project_release'
data_candidates = [Path(DATA_DIR).expanduser()] if DATA_DIR is not None else [
    project, release, release / 'project_release', project.parent, project.parent / 'project_release'
]
data_path = next((folder for folder in data_candidates
                  if all((folder / name).is_file() for name in ('catalog.json', 'train_gold.json'))), None)
if data_path is None:
    raise FileNotFoundError('첫 셀의 DATA_DIR에 catalog.json과 train_gold.json이 함께 있는 폴더를 지정하세요.')
CATALOG_PATH = data_path / 'catalog.json'

# 검색기와 재랭커에 실제 학습 때와 같은 필드 순서와 구분자를 제공한다.
def metadata_text(item):
    fields = (('title', 'Product'), ('store', 'Store'), ('features', 'Features'), ('description', 'Description'))
    parts = []
    for field, label in fields:
        value = item[field].strip()
        if value:
            parts.append(f'{label}: {value}')
    return ' [SEP] '.join(parts) or 'Product metadata not available'


# FAISS 인덱스의 상품 행 번호와 원본 JSON의 상품 순서를 맞춘다.
catalog_rows = read_json(CATALOG_PATH)
ITEM_IDS = [item['item_id'] for item in catalog_rows]
ITEM_TO_IDX = {item_id: idx for idx, item_id in enumerate(ITEM_IDS)}
if not ITEM_IDS or len(ITEM_TO_IDX) != len(ITEM_IDS):
    raise ValueError('catalog.json에 상품이 없거나 item_id가 중복됐습니다.')
CAT_TEXTS = [metadata_text(item) for item in catalog_rows]
del catalog_rows
gold_rows = read_json(data_path / 'train_gold.json')
gold_by_query = {row['query_id']: row for row in gold_rows}
if len(gold_by_query) != len(gold_rows):
    raise ValueError('train_gold.json의 query_id가 중복됐습니다.')

split_file = RUN_DIR / 'split.json'
if split_file.is_file():
    # 저장된 VAL의 순서까지 복원해 같은 VAL_LIMIT에서 같은 쿼리를 비교한다.
    split = read_json(split_file)
    val_query_ids = split['val_query_ids']
    if len(set(val_query_ids)) != len(val_query_ids) or set(val_query_ids) - gold_by_query.keys():
        raise ValueError('split.json의 VAL query_id가 train_gold.json과 일치하지 않습니다.')
    SEED = int(split.get('seed', SPLIT_SEED))
    split_source = str(split_file)
else:
    # 원래 학습 코드와 같은 방식으로 먼저 상품 ID를 분할한다.
    SEED = SPLIT_SEED
    unique_items = sorted({row['item_id'] for row in gold_rows})
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_items)
    n_val = max(1, int(round(len(unique_items) * CFG['val_fraction'])))
    val_items = set(unique_items[:n_val])
    val_query_ids = [row['query_id'] for row in gold_rows if row['item_id'] in val_items]
    split_source = f'상품 ID 기준 재현 (seed={SEED}, val_fraction={CFG["val_fraction"]})'

if VAL_LIMIT is not None and VAL_LIMIT < 1:
    raise ValueError('VAL_LIMIT은 양의 정수 또는 None이어야 합니다.')
VAL = []
for query_id in val_query_ids[:VAL_LIMIT]:
    row = gold_by_query[query_id]
    reviews = [review['text'].strip() for review in row['reviews']]
    if len(reviews) != 5 or row['item_id'] not in ITEM_TO_IDX:
        raise ValueError(f'잘못된 gold 리뷰 묶음 또는 카탈로그에 없는 상품: {query_id}')
    VAL.append({'query_id': query_id, 'item_id': row['item_id'], 'reviews': reviews})
if not VAL:
    raise ValueError('평가할 VAL 쿼리가 없습니다.')
del gold_rows, gold_by_query
DIAGNOSTICS_DIR = RUN_DIR / 'retrieval_diagnostics'
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

print('장치:', DEVICE, '| 검색기:', COLBERT_PATH)
print('재랭커:', CROSS_PATH if EVAL_CROSS else '검색기만 평가')
print('설정:', settings_source, '| VAL:', len(VAL), '건 /', split_source)

## 체크포인트와 상품 인덱스

검색기는 체크포인트에서 리뷰·상품 인코더와 토큰 투영 가중치를 복원합니다. 재랭커는 해당 단계의 가중치를 복원합니다. `EXISTING_INDEX_DIR`을 지정할 때는 **선택한 검색기 체크포인트와 같은 카탈로그 순서로 생성한 인덱스**를 사용하세요.

기본 진단용 인덱스는 체크포인트·카탈로그·상품 토큰 설정을 기준으로 저장합니다. `metadata_tokens`를 바꾸면 그 길이에 맞는 인덱스를 새로 준비하며, `ann_pool`·`ann_per_token`·`candidate_topk`를 바꾸는 실험은 같은 상품 벡터를 쓸 수 있습니다.

In [ ]:
# 3. 학습 체크포인트의 구조를 복원한다. 가중치는 .pt에서, 백본 구조와 토크나이저는 학습 설정에서 읽는다.
class ColBERTRetriever(nn.Module):
    def __init__(self, config, dimension):
        super().__init__()
        self.query_encoder = AutoModel.from_config(config)
        self.doc_encoder = AutoModel.from_config(config)
        self.query_proj = nn.Linear(config.hidden_size, dimension, bias=False)
        self.doc_proj = nn.Linear(config.hidden_size, dimension, bias=False)

    # 학습 노트북과 같은 정규화와 유효 토큰 마스크를 사용해 MaxSim용 벡터를 반환한다.
    def encode(self, tokens, side):
        backbone = self.query_encoder if side == 'query' else self.doc_encoder
        projection = self.query_proj if side == 'query' else self.doc_proj
        hidden = backbone(**tokens).last_hidden_state
        mask = tokens['attention_mask'].bool()
        special = torch.zeros_like(mask)
        for token_id in SPECIAL_IDS:
            special |= tokens['input_ids'].eq(token_id)
        valid = mask & ~special
        empty = ~valid.any(dim=-1)
        if empty.any():
            valid[empty] = mask[empty]
        return F.normalize(projection(hidden), p=2, dim=-1), valid


class CrossReranker(nn.Module):
    def __init__(self, config):
        super().__init__()
        config.num_labels = 1
        self.net = AutoModelForSequenceClassification.from_config(config)

    # 리뷰 다섯 개를 합친 문장과 상품 텍스트를 한 쌍으로 채점한다.
    def score(self, reviews, doc_texts):
        tokens = TOKENIZER(reviews, doc_texts, padding=True, truncation='longest_first',
                           max_length=CFG['cross_tokens'], return_tensors='pt')
        tokens = {name: value.to(DEVICE) for name, value in tokens.items()}
        return self.net(**tokens).logits.reshape(-1)


def load_retriever(path, config):
    # 보조 속성 헤드와 pair loss 파라미터는 검색 점수에 쓰이지 않는다.
    # 인코더와 투영층의 키·크기가 모두 맞는지 strict 로딩으로 검사한다.
    state = torch.load(path, map_location='cpu', weights_only=True)
    if 'query_proj.weight' not in state:
        raise ValueError(f'ColBERT 체크포인트에 query_proj.weight가 없습니다. {path}')
    dimension = int(state['query_proj.weight'].shape[0])
    model = ColBERTRetriever(config, dimension)
    required = ('query_encoder.', 'doc_encoder.', 'query_proj.', 'doc_proj.')
    model.load_state_dict({key: value for key, value in state.items() if key.startswith(required)}, strict=True)
    return model.to(DEVICE).eval(), dimension


def load_cross(path, config):
    model = CrossReranker(config)
    state = torch.load(path, map_location='cpu', weights_only=True)
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()


model_config = AutoConfig.from_pretrained(CFG['backbone'])
TOKENIZER = AutoTokenizer.from_pretrained(CFG['backbone'], use_fast=True)
SPECIAL_IDS = set(TOKENIZER.all_special_ids)
retriever, CFG['dimension'] = load_retriever(COLBERT_PATH, model_config)
reranker = load_cross(CROSS_PATH, AutoConfig.from_pretrained(CFG['backbone'])) if EVAL_CROSS else None


def tokenize(texts,max_length):
    out=TOKENIZER(texts,padding=True,truncation=True,max_length=max_length,return_tensors='pt')
    return {k:v.to(DEVICE) for k,v in out.items()}


def joined_reviews(reviews):
    # 리뷰별 상한을 먼저 적용한다. 재랭커가 상품과 합쳐 다시 자를 때 뒤쪽 리뷰가 더 짧아질 수 있다.
    chunks=[]
    for i,r in enumerate(reviews):
        ids=TOKENIZER.encode(r,add_special_tokens=False,truncation=True,
                             max_length=CFG['cross_tokens_per_review'])
        chunks.append(f'Review {i+1}: '+TOKENIZER.decode(ids,skip_special_tokens=True))
    return ' [SEP] '.join(chunks)

In [ ]:
# 4. 상품 벡터와 FAISS 인덱스를 준비한다. 상품을 인코딩하는 순서는 catalog.json의 행 순서다.
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


@torch.inference_mode()
def build_token_index(model, folder):
    folder.mkdir(parents=True)
    n_items, length, dim = len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension']
    emb = np.lib.format.open_memmap(folder / 'doc_embeddings.npy', mode='w+', dtype=np.float16,
                                    shape=(n_items, length, dim))
    mask = np.lib.format.open_memmap(folder / 'doc_masks.npy', mode='w+', dtype=np.bool_,
                                     shape=(n_items, length))
    batch = CFG['index_doc_batch']
    for start in tqdm(range(0, n_items, batch), desc='Encode catalog'):
        end = min(n_items, start + batch)
        tokens = tokenize(CAT_TEXTS[start:end], length)
        with amp_context():
            reps, valid = model.encode(tokens, 'doc')
        width = reps.shape[1]
        emb[start:end] = 0
        mask[start:end] = False
        emb[start:end, :width] = reps.float().cpu().numpy().astype(np.float16)
        mask[start:end, :width] = valid.cpu().numpy()
    emb.flush()
    mask.flush()

    # 학습 노트북과 같은 방식으로 최대 12만 유효 토큰을 골라 IVF의 군집을 구성한다.
    valid_docs, valid_positions = np.where(mask)
    n_train = min(120_000, len(valid_docs))
    sample = np.random.default_rng(SEED).choice(len(valid_docs), size=n_train, replace=False)
    train_vectors = np.ascontiguousarray(emb[valid_docs[sample], valid_positions[sample]], dtype=np.float32)
    token2doc = valid_docs.astype(np.int32)
    del valid_docs, valid_positions
    nlist = min(CFG['ann_nlist'], max(1, n_train // 40))
    index = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, nlist, faiss.METRIC_INNER_PRODUCT)
    index.train(train_vectors)
    index.nprobe = min(CFG['ann_nprobe'], nlist)
    batch = CFG['index_add_docs']
    for start in tqdm(range(0, n_items, batch), desc='Build token ANN'):
        end = min(n_items, start + batch)
        valid = mask[start:end].reshape(-1)
        vectors = emb[start:end].reshape(-1, dim)[valid]
        if len(vectors):
            index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    assert index.ntotal == len(token2doc)
    faiss.write_index(index, str(folder / 'token_ivf.index'))
    np.save(folder / 'token_to_item.npy', token2doc)


def open_token_index(folder):
    # 같은 인덱스의 네 파일을 읽고 상품 수·토큰 길이·임베딩 차원이 맞는지 확인한다.
    emb = np.load(folder / 'doc_embeddings.npy', mmap_mode='r')
    mask = np.load(folder / 'doc_masks.npy', mmap_mode='r')
    token2doc = np.load(folder / 'token_to_item.npy', mmap_mode='r')
    index = faiss.read_index(str(folder / 'token_ivf.index'))
    expected = (len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension'])
    if (emb.shape != expected or mask.shape != expected[:2] or mask.dtype != np.bool_
            or index.d != expected[2] or token2doc.ndim != 1 or index.ntotal != len(token2doc)):
        raise ValueError(f'상품 인덱스의 크기가 현재 설정과 맞지 않습니다. {folder}')
    if not len(token2doc) or token2doc.min() < 0 or token2doc.max() >= len(ITEM_IDS):
        raise ValueError(f'인덱스의 토큰-상품 대응표를 확인하세요. {folder}')
    index.nprobe = min(CFG['ann_nprobe'], index.nlist)
    return {'index': index, 'token2doc': token2doc, 'emb': emb, 'mask': mask}


if EXISTING_INDEX_DIR is not None:
    index_path = Path(EXISTING_INDEX_DIR).expanduser().resolve()
    index_source = '지정한 기존 인덱스'
else:
    # 체크포인트나 상품 입력이 달라지면 별도 인덱스를 만든다. 검색 후보 수 수정은 같은 인덱스를 사용한다.
    signature = {
        'checkpoint': file_sha256(COLBERT_PATH),
        'catalog': file_sha256(CATALOG_PATH),
        'seed': SEED,
        'precision': 'bfloat16' if USE_BFLOAT16 else 'float32',
        'input': {key: CFG[key] for key in ('backbone', 'metadata_tokens', 'dimension', 'ann_nlist',
                                           'index_doc_batch', 'index_add_docs')},
    }
    index_key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:20]
    cache = DIAGNOSTICS_DIR / 'index_cache'
    cache.mkdir(exist_ok=True)
    index_path = cache / index_key
    index_source = '진단용 캐시'
    if not index_path.exists():
        # 네 파일의 저장을 끝낸 인덱스만 캐시 위치에 옮긴다.
        with TemporaryDirectory(dir=cache) as staging:
            staged_index = Path(staging) / 'index'
            build_token_index(retriever, staged_index)
            write_json(staged_index / 'signature.json', signature)
            try:
                staged_index.rename(index_path)
            except OSError:
                if not (index_path / 'signature.json').is_file():
                    raise
        index_source = '체크포인트에서 생성한 진단용 인덱스'
index_bundle = open_token_index(index_path)
print(index_source, '| 상품:', len(ITEM_IDS), '| 인덱스 토큰:', index_bundle['index'].ntotal)
print('인덱스 경로:', index_path)

## 검증 쿼리별 탈락 지점

| `outcome` | 판정 |
|---|---|
| `ann_no_token_hit` | 정답 상품의 토큰이 FAISS 이웃에 한 번도 나오지 않았습니다. |
| `ann_pool_cutoff` | 정답 상품에 토큰 히트가 있었지만 근사 순위로 뽑은 상품 풀에서 빠졌습니다. |
| `maxsim_cutoff` | 정답 상품을 MaxSim으로 계산했으나 재랭킹 후보 Top K 밖이었습니다. |
| `cross_cutoff` | 정답 상품이 재랭킹 후보에 들어왔으나 최종 Top 10 밖이었습니다. |
| `hit_final` / `hit_candidates` | 최종 Top 10에 도달했거나, 검색기만 평가한 실행에서 재랭킹 후보에 들었습니다. |

토큰 이웃에서 상품 후보를 충분히 모으지 못했을 때는 기존 코드처럼 카탈로그 순서로 보충합니다. `gold_in_ann_pool`은 보충 **전**, `gold_in_scored_pool`은 보충 **후**의 정답 포함 여부입니다.

정답이 실제 채점 풀 밖에 있으면 **정답 하나에 대해서만 별도로 MaxSim을 계산**합니다. `gold_minus_cutoff`가 양수면 현재 후보를 놓고 봤을 때 정답이 기존 K위보다 높았다는 뜻입니다. 이 비교는 현재 후보에 대한 것이며, 전체 카탈로그의 정확 순위와는 별개입니다.

In [ ]:
# 5. 검색 점수는 원본과 같은 방식으로 계산하고, 각 중간 순위와 정답 점수를 함께 남긴다.
@torch.inference_mode()
def encode_five_reviews(model, reviews):
    tokens = tokenize(reviews, CFG['query_tokens'])
    with amp_context():
        embeddings, valid = model.encode(tokens, 'query')
    return embeddings.float().cpu().numpy(), valid.cpu().numpy()


def token_hit_order(query, query_mask, index_bundle, per_token):
    # 동일 리뷰 토큰이 한 상품의 여러 토큰을 찾으면 그중 가장 큰 내적만 사용한다.
    token_vectors = np.ascontiguousarray(query[query_mask], dtype=np.float32)
    distances, token_ids = index_bundle['index'].search(token_vectors, per_token)
    token2doc = index_bundle['token2doc']
    rough = defaultdict(float)
    for similarities, hits in zip(distances, token_ids):
        best = {}
        for similarity, token_id in zip(similarities, hits):
            if token_id < 0:
                continue
            doc = int(token2doc[token_id])
            best[doc] = max(best.get(doc, -1e6), float(similarity))
        for doc, similarity in best.items():
            rough[doc] += max(0.0, similarity)
    return sorted(rough, key=lambda doc: (-rough[doc], doc))


@torch.inference_mode()
def maxsim_scores(query, query_mask, index_bundle, doc_indices):
    # 후보의 유효 상품 토큰 전체를 읽고, 리뷰 토큰마다 가장 높은 내적을 더한다.
    # 최종 점수 하나는 다섯 리뷰의 MaxSim 점수 평균이다.
    qe = torch.as_tensor(query, device=DEVICE)
    qm = torch.as_tensor(query_mask, device=DEVICE)
    scores = []
    for start in range(0, len(doc_indices), 96):
        rows = doc_indices[start:start + 96]
        de = torch.as_tensor(np.asarray(index_bundle['emb'][rows]).astype(np.float32), device=DEVICE)
        dm = torch.as_tensor(np.asarray(index_bundle['mask'][rows]), device=DEVICE)
        with amp_context():
            simil = torch.einsum('rqd,cmd->rcqm', qe, de)
            simil = simil.masked_fill(~dm[None, :, None, :], -1e4)
            review_scores = (simil.max(-1).values * qm[:, None, :]).sum(-1)
        scores.append(review_scores.mean(0).float().cpu().numpy())
    return np.concatenate(scores)


@torch.inference_mode()
def rerank_candidate_ids(model,reviews,ids):
    model.eval()
    q=joined_reviews(reviews)
    result=[]
    for start in range(0,len(ids),CFG['cross_infer_batch']):
        sub=ids[start:start+CFG['cross_infer_batch']]
        with amp_context():
            logits=model.score([q]*len(sub),[CAT_TEXTS[ITEM_TO_IDX[iid]] for iid in sub])
        result.extend(logits.float().cpu().numpy().tolist())
    order=np.argsort(-np.asarray(result),kind='stable')
    return [ids[i] for i in order],[result[i] for i in order]


def diagnose_query(row, query, query_mask, index_bundle, setting, cross):
    gold = ITEM_TO_IDX[row['item_id']]
    topk = min(setting['candidate_topk'], len(ITEM_IDS))
    ann_order = token_hit_order(query, query_mask, index_bundle, setting['ann_per_token'])
    ann_rank = ann_order.index(gold) + 1 if gold in ann_order else None
    ann_pool = ann_order[:setting['ann_pool']]
    scored_pool = ann_pool.copy()
    if len(scored_pool) < topk:
        seen = set(scored_pool)
        need = min(len(ITEM_IDS), max(topk, setting['ann_pool'])) - len(scored_pool)
        scored_pool.extend(islice((idx for idx in range(len(ITEM_IDS)) if idx not in seen), need))

    # 정답을 강제로 채점한 점수는 진단에만 기록한다. 재랭킹 후보는 실제 검색 결과로 만든다.
    values = maxsim_scores(query, query_mask, index_bundle, scored_pool)
    order = np.argsort(-values, kind='stable')
    exact_order = [scored_pool[i] for i in order]
    candidate_ids = [ITEM_IDS[idx] for idx in exact_order[:topk]]
    exact_rank = exact_order.index(gold) + 1 if gold in exact_order else None
    gold_score = (float(values[order[exact_rank - 1]]) if exact_rank is not None
                  else float(maxsim_scores(query, query_mask, index_bundle, [gold])[0]))
    cutoff = float(values[order[topk - 1]])
    gap = gold_score - cutoff
    forced_gold = None
    if exact_rank is None:
        if np.isclose(gold_score, cutoff, rtol=1e-6, atol=1e-6):
            forced_gold = 'tied_cutoff'
        else:
            forced_gold = 'above_cutoff' if gap > 0 else 'below_cutoff'

    # 학습 노트북의 predict_one과 같이 같은 후보를 cross-encoder에 넘긴다.
    cross_rank = None
    cross_top_item = None
    final_hit = None
    if cross is not None:
        reranked, _ = rerank_candidate_ids(cross, row['reviews'], candidate_ids)
        cross_top_item = reranked[0]
        if row['item_id'] in reranked:
            cross_rank = reranked.index(row['item_id']) + 1
        final_hit = cross_rank is not None and cross_rank <= 10

    candidate_hit = exact_rank is not None and exact_rank <= topk
    if candidate_hit:
        outcome = 'hit_candidates' if cross is None else ('hit_final' if final_hit else 'cross_cutoff')
    elif exact_rank is not None:
        outcome = 'maxsim_cutoff'
    elif ann_rank is not None:
        outcome = 'ann_pool_cutoff'
    else:
        outcome = 'ann_no_token_hit'

    return {
        'setting': setting['name'],
        'query_id': row['query_id'],
        'gold_item_id': row['item_id'],
        'ann_token_hit_items': len(ann_order),
        'gold_ann_rank': ann_rank,
        'gold_in_ann_pool': gold in ann_pool,
        'gold_in_scored_pool': exact_rank is not None,
        'scored_pool_size': len(scored_pool),
        'fallback_items': len(scored_pool) - len(ann_pool),
        'gold_maxsim_rank': exact_rank,
        'gold_maxsim_score': gold_score,
        'topk_cutoff_score': cutoff,
        'gold_minus_cutoff': gap,
        'forced_gold': forced_gold,
        'retriever_hit_at_10': row['item_id'] in candidate_ids[:10],
        'retriever_hit_at_k': candidate_hit,
        'retriever_top_item_id': candidate_ids[0],
        'cross_gold_rank': cross_rank,
        'cross_top_item_id': cross_top_item,
        'final_hit_at_10': final_hit,
        'outcome': outcome,
    }

In [ ]:
# 6. 고정된 VAL에서 평가한다. 비교 설정을 켜면 리뷰 임베딩을 한 번 계산해 검색 실험에서 공유한다.
settings = [{'name': 'baseline', 'ann_per_token': CFG['ann_per_token'],
             'ann_pool': CFG['ann_pool'], 'candidate_topk': CFG['candidate_topk']}]
if RUN_ANN_SWEEP:
    base = settings[0]
    settings += [
        {**base, 'name': 'more_token_neighbors', 'ann_per_token': 2 * base['ann_per_token']},
        {**base, 'name': 'larger_ann_pool', 'ann_pool': 2 * base['ann_pool']},
    ]
search_keys = ('ann_per_token', 'ann_pool', 'candidate_topk')
if any(not isinstance(setting[key], int) or setting[key] < 1 for setting in settings for key in search_keys):
    raise ValueError('ann_per_token, ann_pool, candidate_topk는 양의 정수여야 합니다.')

records = []
for row in tqdm(VAL, desc='Diagnose VAL'):
    query, query_mask = encode_five_reviews(retriever, row['reviews'])
    for setting in settings:
        cross = reranker if setting['name'] == 'baseline' else None
        records.append(diagnose_query(row, query, query_mask, index_bundle, setting, cross))
query_results = pd.DataFrame(records)


def summarize_run(frame, setting):
    # 검색 단계별로 정답을 담은 쿼리의 비율과 정답이 처음 탈락한 건수를 계산한다.
    outcome_counts = frame['outcome'].value_counts()
    ranked = frame['gold_maxsim_rank'].fillna(float('inf'))
    cross_evaluated = frame['final_hit_at_10'].dropna()
    candidate_hits = int(frame['retriever_hit_at_k'].sum())
    final_hits = int(cross_evaluated.sum()) if len(cross_evaluated) else None
    return {
        'setting': setting['name'],
        'ann_per_token': setting['ann_per_token'],
        'ann_pool': setting['ann_pool'],
        'candidate_topk': setting['candidate_topk'],
        'n_queries': len(frame),
        'ann_token_hit_rate': float(frame['gold_ann_rank'].notna().mean()),
        'ann_pool_recall': float(frame['gold_in_ann_pool'].mean()),
        'retriever_recall_at_10': float(frame['retriever_hit_at_10'].mean()),
        'candidate_recall_at_k': float(frame['retriever_hit_at_k'].mean()),
        'maxsim_recall_at_100_in_pool': float((ranked <= 100).mean()),
        'maxsim_recall_at_200_in_pool': float((ranked <= 200).mean()),
        'final_recall_at_10': final_hits / len(frame) if final_hits is not None else None,
        'cross_hit_given_candidate': (final_hits / candidate_hits
                                      if final_hits is not None and candidate_hits else None),
        'ann_no_token_hit': int(outcome_counts.get('ann_no_token_hit', 0)),
        'ann_pool_cutoff': int(outcome_counts.get('ann_pool_cutoff', 0)),
        'maxsim_cutoff': int(outcome_counts.get('maxsim_cutoff', 0)),
        'cross_cutoff': int(outcome_counts.get('cross_cutoff', 0)) if len(cross_evaluated) else None,
        'forced_gold_above_cutoff': int(frame['forced_gold'].eq('above_cutoff').sum()),
        'forced_gold_below_cutoff': int(frame['forced_gold'].eq('below_cutoff').sum()),
        'forced_gold_tied_cutoff': int(frame['forced_gold'].eq('tied_cutoff').sum()),
    }


stage_summary = pd.DataFrame([
    summarize_run(query_results.loc[query_results['setting'].eq(setting['name'])], setting)
    for setting in settings
])
report_dir = DIAGNOSTICS_DIR / (
    f'{CHECKPOINT_STAGE}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}'
)
report_dir.mkdir()
stage_summary.to_csv(report_dir / 'stage_summary.csv', index=False)
query_results.to_csv(report_dir / 'query_diagnostics.csv', index=False)
write_json(report_dir / 'settings.json', {
    'colbert_checkpoint': str(COLBERT_PATH.resolve()),
    'cross_checkpoint': str(CROSS_PATH.resolve()) if reranker is not None else None,
    'catalog': str(CATALOG_PATH.resolve()),
    'index_path': str(index_path),
    'split_source': split_source,
    'settings_source': settings_source,
    'n_queries': len(VAL),
    'config': CFG,
    'search_settings': settings,
})
print('진단 결과 경로:', report_dir)
display(stage_summary[['setting', 'n_queries', 'ann_per_token', 'ann_pool', 'candidate_topk',
                       'ann_token_hit_rate', 'ann_pool_recall', 'retriever_recall_at_10',
                       'candidate_recall_at_k', 'maxsim_recall_at_100_in_pool', 'final_recall_at_10']].round(4))
display(stage_summary[['setting', 'ann_no_token_hit', 'ann_pool_cutoff', 'maxsim_cutoff',
                       'cross_cutoff', 'forced_gold_above_cutoff', 'forced_gold_below_cutoff',
                       'forced_gold_tied_cutoff']])

# 정답이 탈락한 예시의 상품 ID, 근사 순위, MaxSim 순위와 점수 차이를 살펴본다.
failed = query_results.loc[query_results['setting'].eq('baseline')
                           & ~query_results['outcome'].isin(['hit_final', 'hit_candidates'])]
display(failed[['query_id', 'gold_item_id', 'outcome', 'gold_ann_rank', 'gold_maxsim_rank',
                'gold_minus_cutoff', 'cross_gold_rank', 'retriever_top_item_id']].head(15))

## 결과로 결정할 실험

- **토큰 이웃 누락이 많으면** `RUN_ANN_SWEEP = True`에서 `more_token_neighbors`의 후보 Recall을 확인합니다. **근사 풀에서 탈락한 건이 많으면** `larger_ann_pool`을 확인합니다. 비교 설정은 검색기만 평가하며, 최종 Recall을 확인하려면 원하는 값을 첫 셀의 `CFG_OVERRIDES`에 넣어 baseline으로 다시 평가합니다.
- **MaxSim에서 탈락한 건이 많으면** `gold_maxsim_rank`와 `maxsim_recall_at_100_in_pool`을 봅니다. 51~100위에 정답이 모였다면 그 범위까지 재랭킹하는 실험을 할 수 있습니다. 순위가 계속 낮다면 토큰화 후 구별 단서 또는 학습 때의 hard negative를 살펴봅니다.
- **재랭킹 오답이 많으면** `cross_cutoff` 사례를 살펴보고 실제 새 후보를 학습용 hard negative로 쓸지 판단할 수 있습니다. 체크포인트를 변경한 후에도 현재 cross-encoder를 불러와 같은 VAL로 우선 평가할 수 있습니다.

`query_diagnostics.csv`에는 쿼리별 정답 ID와 순위·점수, `stage_summary.csv`에는 설정별 집계, `settings.json`에는 평가에 사용한 파일과 설정이 저장됩니다.